# TP2 — Régularisation en Machine Learning
**Étudiante :** Sara El Ghayati  
**github** : https://github.com/saragh66/tp_technique_ia
---

## Objectifs du TP
Ce TP explore l'effet de la **régularisation** (L1, L2 et aucune) sur des modèles de régression logistique avec des features polynomiales.
Nous travaillons sur deux jeux de données :
- **Partie 1 :** `make_moons` — données non-linéaires en 2D
- **Partie 2 :** `make_classification` — données haute dimension (50 features) réduites par PCA


## 1. Importation des bibliothèques

On regroupe **tous les imports** dans une seule cellule pour éviter les erreurs de type `NameError`.
L'erreur principale du notebook original était l'absence de `from sklearn.pipeline import Pipeline`.


In [ ]:
# ============================================================
#  TOUS LES IMPORTS REGROUPÉS ICI — correction de NameError
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline                          # <-- manquait dans l'original
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA

# Style global des graphiques
plt.rcParams.update({
    'axes.facecolor':  '#1a0033',
    'figure.facecolor':'#1a0033',
    'axes.edgecolor':  '#cc66ff',
    'axes.labelcolor': '#cc66ff',
    'xtick.color':     '#cc66ff',
    'ytick.color':     '#cc66ff',
    'text.color':      '#cc66ff',
    'legend.facecolor':'#2d0057',
    'legend.edgecolor':'#cc66ff',
    'axes.titlecolor': '#ff66cc',
})

print("✅ Tous les imports sont chargés avec succès.")

---
## PARTIE 1 — Dataset `make_moons` (2D)

### 2. Génération et visualisation des données

`make_moons` génère deux classes en forme de croissants entrelacés.  
Ce jeu de données est idéal pour tester des frontières de décision **non-linéaires**.

| Paramètre | Valeur | Signification |
|-----------|--------|---------------|
| `n_samples` | 150 | Nombre total d'échantillons |
| `noise` | 0.30 | Niveau de bruit ajouté |
| `test_size` | 0.30 | 30% pour le test |


In [ ]:
# Génération du dataset make_moons
X, y = make_moons(n_samples=150, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Taille train : {X_train.shape}  |  Taille test : {X_test.shape}")

In [ ]:
# Visualisation des données make_moons
fig = plt.figure(figsize=(8, 5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='plasma',
            edgecolors='white', linewidths=0.5, label='Train data')
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='plasma',
            marker='x', alpha=0.8, label='Test data')
plt.legend()
plt.title('Sara El Ghayati — Data Visualization (make_moons)',
          fontsize=14, fontweight='bold', color='#ff66cc')
plt.tight_layout()
plt.show()

**Interprétation :** On observe deux classes entrelacées qui ne sont pas séparables linéairement.  
Les points **ronds** représentent les données d'entraînement, les **croix** les données de test.  
La couleur encode la classe (0 ou 1). Un modèle avec features polynomiales sera nécessaire pour capturer cette frontière courbe.


### 3. Frontières de décision — Comparaison des régularisations (Degree 10)

On entraîne **3 modèles** de régression logistique avec des features polynomiales de degré 10 :

| Modèle | Régularisation | Solver | Paramètre C |
|--------|---------------|--------|-------------|
| `model_none` | Aucune (`penalty=None`) | `lbfgs` | — |
| `model_l1` | L1 (Lasso) | `liblinear` | 0.1 |
| `model_l2` | L2 (Ridge) | `lbfgs` | 0.1 |

Chaque pipeline contient : **PolynomialFeatures → StandardScaler → LogisticRegression**


In [ ]:
# Degré polynomial
degree = 10

# Modèle 1 : Aucune régularisation
model_none = Pipeline([
    ('poly',    PolynomialFeatures(degree=degree)),
    ('scaler',  StandardScaler()),
    ('log_reg', LogisticRegression(penalty=None, solver='lbfgs', max_iter=5000))
])

# Modèle 2 : Régularisation L1
model_l1 = Pipeline([
    ('poly',    PolynomialFeatures(degree=degree)),
    ('scaler',  StandardScaler()),
    ('log_reg', LogisticRegression(penalty='l1', solver='liblinear', C=0.1, max_iter=5000))
])

# Modèle 3 : Régularisation L2
model_l2 = Pipeline([
    ('poly',    PolynomialFeatures(degree=degree)),
    ('scaler',  StandardScaler()),
    ('log_reg', LogisticRegression(penalty='l2', solver='lbfgs', C=0.1, max_iter=5000))
])

# Entraînement
model_none.fit(X_train, y_train)
model_l1.fit(X_train, y_train)
model_l2.fit(X_train, y_train)

# Fonction de tracé de la frontière de décision
def plot_decision_boundary(model, ax, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, 0.02),
        np.arange(y_min, y_max, 0.02)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.4, cmap='plasma')
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='plasma',
               edgecolors='white', linewidths=0.5, marker='o', label='Train')
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='plasma',
               edgecolors='white', linewidths=0.5, marker='^', alpha=0.7, label='Test')
    ax.set_title(title)
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")
    ax.legend()

# Affichage côte à côte
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#F5F0FF')

plot_decision_boundary(model_none, axes[0], f"No Regularization (Degree {degree})")
plot_decision_boundary(model_l1,   axes[1], f"L1 Regularization (Degree {degree})")
plot_decision_boundary(model_l2,   axes[2], f"L2 Regularization (Degree {degree})")

plt.suptitle('Sara El Ghayati', fontsize=11, color='#ff66cc', style='italic', x=0.01, ha='left')
plt.tight_layout()
plt.show()

# Résumé des performances
for name, model in [('No Reg', model_none), ('L1', model_l1), ('L2', model_l2)]:
    tr = accuracy_score(y_train, model.predict(X_train))
    te = accuracy_score(y_test,  model.predict(X_test))
    print(f"{name:8s} — Train: {tr:.3f}  |  Test: {te:.3f}")

**Interprétation des frontières de décision :**

- **Sans régularisation** : La frontière est très irrégulière et sinueuse — le modèle **surappend (overfitting)** les données d'entraînement. Il mémorise le bruit plutôt que d'apprendre la structure générale.
- **Régularisation L1** : La frontière est plus lisse. La pénalité L1 met certains coefficients exactement à zéro (sélection de features sparse), ce qui simplifie le modèle.
- **Régularisation L2** : Frontière également lisse. La pénalité L2 réduit uniformément tous les coefficients sans les annuler complètement, produisant un modèle plus régulier et généralisable.

> 💡 **Conclusion** : Avec un degré polynomial élevé (10), la régularisation est **indispensable** pour éviter le surapprentissage.


---
## PARTIE 2 — Dataset `make_classification` (50 features → PCA 2D)

### 4. Génération des données haute dimension

On génère un dataset plus réaliste avec **50 features** dont seulement 10 sont vraiment informatives.  
On applique une **ACP (PCA)** pour réduire à 2 dimensions afin de pouvoir visualiser les frontières.

| Paramètre | Valeur |
|-----------|--------|
| `n_samples` | 3000 |
| `n_features` | 50 |
| `n_informative` | 10 |
| `n_redundant` | 10 |
| PCA components | 2 |


In [ ]:
# Génération du dataset make_classification
X, y = make_classification(
    n_samples=3000,
    n_features=50,
    n_informative=10,
    n_redundant=10,
    n_classes=2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Réduction PCA à 2 composantes pour visualisation
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)
X_pca       = pca.transform(X)

variance_expliquee = pca.explained_variance_ratio_.sum() * 100
print(f"Variance expliquée par les 2 composantes PCA : {variance_expliquee:.1f}%")

**Note :** Le pourcentage de variance expliquée indique combien d'information est conservée après la réduction à 2D.  
Un faible pourcentage signifie que la PCA perd une partie de l'information — les résultats de visualisation sont donc **indicatifs**, pas définitifs.


### 5. Frontières de décision sur données PCA — Degrés 2 et 3

On compare les 3 types de régularisation pour **deux degrés polynomiaux** (2 et 3) sur les données réduites par PCA.


In [ ]:
degrees = [2, 3]

def create_model(degree, reg_type):
    """Construit un pipeline Poly → Scaler → LogReg selon le type de régularisation."""
    if reg_type == "none":
        return Pipeline([
            ('poly',    PolynomialFeatures(degree=degree, include_bias=False)),
            ('scaler',  StandardScaler()),
            ('log_reg', LogisticRegression(penalty=None, solver='lbfgs', max_iter=5000))
        ])
    elif reg_type == "l1":
        return Pipeline([
            ('poly',    PolynomialFeatures(degree=degree, include_bias=False)),
            ('scaler',  StandardScaler()),
            ('log_reg', LogisticRegression(penalty='l1', solver='liblinear', C=0.1, max_iter=5000))
        ])
    elif reg_type == "l2":
        return Pipeline([
            ('poly',    PolynomialFeatures(degree=degree, include_bias=False)),
            ('scaler',  StandardScaler()),
            ('log_reg', LogisticRegression(penalty='l2', solver='lbfgs', C=0.1, max_iter=5000))
        ])

def plot_decision_boundary_pca(model, ax, title):
    """Trace la frontière de décision dans l'espace PCA 2D."""
    x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
    y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, 0.05),
        np.arange(y_min, y_max, 0.05)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='plasma')
    ax.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train, cmap='plasma',
               edgecolors='white', linewidths=0.5, s=10, label='Train')
    ax.scatter(X_test_pca[:, 0], X_test_pca[:, 1], c=y_test, cmap='plasma',
               marker='^', s=10, alpha=0.7, label='Test')
    ax.set_title(title)
    ax.set_xlabel("PCA 1")
    ax.set_ylabel("PCA 2")

# Grille 2 x 3 : 2 degrés × 3 régularisations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor('#F5F0FF')

for i, degree in enumerate(degrees):
    for j, (reg, label) in enumerate([('none', 'No Reg'), ('l1', 'L1'), ('l2', 'L2')]):
        model = create_model(degree, reg)
        model.fit(X_train_pca, y_train)
        plot_decision_boundary_pca(model, axes[i, j], f"Degree {degree} — {label}")

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right')
plt.suptitle('Sara El Ghayati', fontsize=10, color='#8A2BE2', style='italic', x=0.01, ha='left')
plt.tight_layout()
plt.show()

**Interprétation :**  
- **Degré 2** : Les frontières restent relativement simples. La différence entre les régularisations est moins marquée.
- **Degré 3** : Des frontières plus complexes apparaissent. Sans régularisation, le risque de surapprentissage augmente.
- La **régularisation L1 et L2** produisent des frontières similaires en apparence, mais L1 tend à créer des angles plus nets (coefficients sparse).


### 6. Comparaison des précisions Train / Test

On mesure la **précision (accuracy)** sur les données d'entraînement et de test pour chaque modèle.  
Un grand écart entre train et test révèle du **surapprentissage (overfitting)**.


In [ ]:
# Recréer les données (make_classification sans PCA pour cet exercice)
X2, y2 = make_classification(
    n_samples=3000, n_features=50,
    n_informative=10, n_redundant=10,
    random_state=42
)
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.3, random_state=42
)

degrees    = [2, 3]
model_names = ['No Reg', 'L1', 'L2']
reg_types   = ['none', 'l1', 'l2']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor('#F5F0FF')

for i, degree in enumerate(degrees):
    train_acc, test_acc = [], []

    for reg in reg_types:
        model = create_model(degree, reg)
        model.fit(X2_train, y2_train)
        train_acc.append(accuracy_score(y2_train, model.predict(X2_train)))
        test_acc.append(accuracy_score(y2_test,  model.predict(X2_test)))

    x = np.arange(len(model_names))
    ax = axes[i]
    ax.plot(x, train_acc, marker='o', label='Train Accuracy', color='#8A2BE2', linewidth=2)
    ax.plot(x, test_acc,  marker='o', label='Test Accuracy',  color='#FF1493', linewidth=2)
    ax.set_xticks(x)
    ax.set_xticklabels(model_names)
    ax.set_title(f"Degree {degree}")
    ax.set_xlabel("Modèle")
    ax.set_ylabel("Accuracy")
    ax.legend(facecolor='#2d0057', edgecolor='#cc66ff', labelcolor='#cc66ff')
    ax.grid(True, color='#4d0099', linestyle='--', alpha=0.5)

    print(f"\n--- Degree {degree} ---")
    for name, tr, te in zip(model_names, train_acc, test_acc):
        gap = tr - te
        print(f"  {name:8s} | Train: {tr:.3f} | Test: {te:.3f} | Gap: {gap:.3f}")

plt.suptitle('Sara El Ghayati', fontsize=10, color='#8A2BE2', style='italic', x=0.01, ha='left')
plt.tight_layout()
plt.show()

**Interprétation :**

- **Sans régularisation** : La précision train est souvent plus élevée que la précision test → signe d'**overfitting** (le modèle mémorise les données d'entraînement).
- **Avec L1 ou L2** : L'écart entre train et test se réduit → meilleure **généralisation**.
- L'effet est plus visible avec un **degré élevé** (plus de features polynomiales = plus de risque d'overfitting).

> 💡 Un bon modèle a des précisions train et test **proches** l'une de l'autre.


### 7. Courbes Train et Test séparées par degré

On affiche les courbes train et test **séparément** pour mieux visualiser l'effet de la régularisation sur chacune.


In [ ]:
degrees     = [2, 3]
model_names = ['No Reg', 'L1', 'L2']
reg_types   = ['none', 'l1', 'l2']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.patch.set_facecolor('#F5F0FF')

for i, degree in enumerate(degrees):
    train_acc, test_acc = [], []

    for reg in reg_types:
        model = create_model(degree, reg)
        model.fit(X2_train, y2_train)
        train_acc.append(accuracy_score(y2_train, model.predict(X2_train)))
        test_acc.append(accuracy_score(y2_test,   model.predict(X2_test)))

    x = np.arange(len(model_names))

    # Courbe Train
    axes[i, 0].plot(x, train_acc, marker='o', label='Train', color='#8A2BE2', linewidth=2)
    axes[i, 0].set_title(f"Degree {degree} — Train Accuracy")
    axes[i, 0].set_xticks(x)
    axes[i, 0].set_xticklabels(model_names)
    axes[i, 0].set_ylabel("Accuracy")
    axes[i, 0].set_ylim(0, 1)
    axes[i, 0].legend(facecolor='#2d0057', edgecolor='#cc66ff', labelcolor='#cc66ff')
    axes[i, 0].grid(True, color='#4d0099', linestyle='--', alpha=0.5)

    # Courbe Test
    axes[i, 1].plot(x, test_acc, marker='o', label='Test', color='#FF1493', linewidth=2)
    axes[i, 1].set_title(f"Degree {degree} — Test Accuracy")
    axes[i, 1].set_xticks(x)
    axes[i, 1].set_xticklabels(model_names)
    axes[i, 1].set_ylabel("Accuracy")
    axes[i, 1].set_ylim(0, 1)
    axes[i, 1].legend(facecolor='#2d0057', edgecolor='#cc66ff', labelcolor='#cc66ff')
    axes[i, 1].grid(True, color='#4d0099', linestyle='--', alpha=0.5)

plt.suptitle('Sara El Ghayati', fontsize=10, color='#8A2BE2', style='italic', x=0.01, ha='left')
plt.tight_layout()
plt.show()

---
## Conclusion Générale

Ce TP a mis en évidence l'importance de la **régularisation** dans les modèles de classification :

| Critère | Sans Régularisation | L1 (Lasso) | L2 (Ridge) |
|---------|--------------------|-----------|-----------|
| Frontière de décision | Très irrégulière | Lisse, sparse | Lisse, uniforme |
| Overfitting | Élevé | Réduit | Réduit |
| Sélection de features | Non | Oui (coeff = 0) | Non |
| Généralisation | Faible | Bonne | Bonne |

**Points clés :**
1. Plus le **degré polynomial** est élevé, plus le modèle est susceptible de surapprendre sans régularisation.
2. La **L1** est utile quand on veut sélectionner automatiquement les features importantes.
3. La **L2** est préférable quand toutes les features contribuent au modèle.
4. Le paramètre **C** contrôle la force de la régularisation : petit C = forte régularisation.
